In [1]:
# Evaluation Metrics for Generative Models: Inception Score, FID, CLIP Score

# Inception Score (IS)

# Fréchet Inception Distance (FID)

# CLIP Score


In [2]:
# Setup

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seed
torch.manual_seed(42)

/Users/girish11/aifromscratch_code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0827 01:18:18.077000 84660 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0827 01:18:18.111000 84660 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0827 01:18:18.135000 84660 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Using device: cpu


In [3]:
# Inception Score Implementation
def load_inception():
    model = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False)
    model.eval()
    model.to(device)
    return model

inception = load_inception()

# The transform_input=False allows us to use our own preprocessing. We need to resize images to 299×299 and normalize according to ImageNet statistics.

inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /Users/girish11/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:04<00:00, 25.2MB/s] 


In [ ]:
# Now implement the Inception Score. We'll compute probabilities in batches, then calculate the score.
def inception_score(images, batch_size=32, splits=10):
    """
    images: list of PIL Images or a tensor of shape [N, 3, H, W] in [0,1] range.
    Returns mean and std of IS over splits.
    """
    if isinstance(images, torch.Tensor):
        # Assume images are already preprocessed and on device
        pass
    else:
        # Convert list of PIL to tensor
        tensors = []
        for img in images:
            img_tensor = inception_transform(img).unsqueeze(0)
            tensors.append(img_tensor)
        images = torch.cat(tensors, dim=0)
    images = images.to(device)

    N = images.size(0)
    preds = []
    with torch.no_grad():
        for i in range(0, N, batch_size):
            batch = images[i:i+batch_size]
            logits = inception(batch)
            probs = F.softmax(logits, dim=1)
            preds.append(probs.cpu())
    preds = torch.cat(preds, dim=0)

    # Split into splits
    split_scores = []
    for k in range(splits):
        part = preds[k * (N // splits): (k+1) * (N // splits)]
        py = part.mean(dim=0)  # marginal p(y)
        kl = part * (torch.log(part + 1e-10) - torch.log(py + 1e-10))
        kl_div = kl.sum(dim=1).mean()
        split_scores.append(torch.exp(kl_div).item())
    return np.mean(split_scores), np.std(split_scores)